# ============================================================================
# TOOL DEFINITIONS - All local operations, no data sent to Gemini
# ============================================================================

In [ ]:
class AnomalyDetectionTools:
    """Local tools for anomaly detection - all data processing happens here"""
    
    @staticmethod
    def load_dataset(filepath: str) -> Dict[str, Any]:
        """Load CSV file and return metadata"""
        try:
            df = pd.read_csv(filepath)[["feature_1","feature_2","feature_3","feature_4","feature_5","feature_6","feature_7","feature_8","feature_9","feature_10"]]
            metadata = {
                "success": True,
                "rows": len(df),
                "columns": list(df.columns),
                "dtypes": df.dtypes.astype(str).to_dict(),
                "missing_values": df.isnull().sum().to_dict(),
                "numeric_columns": df.select_dtypes(include=[np.number]).columns.tolist()
            }
            return {"data": df, "metadata": metadata}
        except FileNotFoundError:
            return {"success": False, "error": "File not found", "data": None}
        except Exception as e:
            return {"success": False, "error": str(e), "data": None}
    
    @staticmethod
    def create_time_splits(df: pd.DataFrame, n_splits: int = 5) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:
        """Create out-of-time validation splits"""
        splits = []
        total_rows = len(df)
        segment_size = total_rows // n_splits
        
        for i in range(n_splits):
            start_idx = i * segment_size
            end_idx = (i + 1) * segment_size if i < n_splits - 1 else total_rows
            segment = df.iloc[start_idx:end_idx]
            
            split_point = int(len(segment) * 0.8)
            train_segment = segment.iloc[:split_point]
            val_segment = segment.iloc[split_point:]
            
            splits.append((train_segment, val_segment))
        
        return splits
    
    @staticmethod
    def calculate_separation_distance(y_pred: np.ndarray, X: np.ndarray) -> float:
        """Calculate Euclidean distance between anomaly and normal centroids"""
        try:
            anomaly_mask = y_pred == 1
            normal_mask = y_pred == 0
            
            if anomaly_mask.sum() == 0 or normal_mask.sum() == 0:
                return 0.0
            
            anomaly_centroid = X[anomaly_mask].mean(axis=0)
            normal_centroid = X[normal_mask].mean(axis=0)
            
            distance = euclidean(anomaly_centroid, normal_centroid)
            return float(distance)
        except Exception as e:
            return 0.0
    
    @staticmethod
    def train_and_evaluate(algorithm: str, params: Dict, splits: List, feature_cols: List) -> Dict:
        """Train model on splits and return average separation distance"""
        distances = []
        errors = []
        
        for fold_idx, (train_df, val_df) in enumerate(splits):
            try:
                X_train = train_df[feature_cols].values
                X_val = val_df[feature_cols].values
                
                # Scale features
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)
                
                # Initialize model
                if algorithm == "IsolationForest":
                    model = IForest(**params)
                elif algorithm == "OCSVM":
                    model = OCSVM(**params)
                elif algorithm == "ABOD":
                    model = ABOD(**params)
                else:
                    return {"success": False, "error": f"Unknown algorithm: {algorithm}"}
                
                # Train and predict
                model.fit(X_train_scaled)
                y_pred = model.predict(X_val_scaled)
                
                # Calculate separation distance
                distance = AnomalyDetectionTools.calculate_separation_distance(y_pred, X_val_scaled)
                distances.append(distance)
                
            except Exception as e:
                errors.append(f"Fold {fold_idx+1}: {str(e)}")
                distances.append(0.0)
        
        avg_distance = np.mean(distances) if distances else 0.0
        
        return {
            "success": len(errors) < len(splits),
            "avg_separation_distance": avg_distance,
            "fold_distances": distances,
            "errors": errors
        }
    
    @staticmethod
    def get_hyperparameter_grid() -> Dict[str, List[Dict]]:
        """Define hyperparameter combinations for each algorithm"""
        return {
            "IsolationForest": [
                {"n_estimators": 100, "max_samples": 256, "contamination": 0.1},
                {"n_estimators": 200, "max_samples": 256, "contamination": 0.1},
                {"n_estimators": 100, "max_samples": 512, "contamination": 0.15},
            ],
            "AutoEncoder": [
                {"hidden_neurons": [64, 32, 16, 32, 64], "epochs": 50, "contamination": 0.1},
                {"hidden_neurons": [128, 64, 32, 64, 128], "epochs": 50, "contamination": 0.1},
                {"hidden_neurons": [64, 32, 16, 32, 64], "epochs": 100, "contamination": 0.1},
            ],
            "OCSVM": [
                {"kernel": "rbf", "nu": 0.1, "gamma": "auto"},
                {"kernel": "rbf", "nu": 0.2, "gamma": "auto"},
                {"kernel": "poly", "nu": 0.1, "gamma": "auto"},
            ],
            "ABOD": [
                {"n_neighbors": 5, "method": "fast"},
                {"n_neighbors": 10, "method": "fast"},
                {"n_neighbors": 15, "method": "fast"},
            ]
        }